In [ ]:
import os
import numpy as np
import pandas as pd
from astropy.io import fits
from scipy import ndimage
from scipy.stats import skew, kurtosis
from skimage.measure import moments, moments_central
import warnings
warnings.filterwarnings('ignore')


#SIMPLE CENTER DETECTION - GALAXIES ARE ALREADY CENTERED


def get_galaxy_center_and_radius(data):
    """
    Since galaxies are already centered in the image, use image center
    Returns: (cy, cx, effective_radius)
    """
    cy = data.shape[0] // 2
    cx = data.shape[1] // 2

    # Estimate effective radius (radius containing 50% of light)
    y, x = np.ogrid[:data.shape[0], :data.shape[1]]
    r = np.sqrt((y - cy)**2 + (x - cx)**2)

    total_flux = np.sum(data)
    r_eff = 0

    for radius in range(5, min(data.shape) // 2):
        mask = r < radius
        if np.sum(data[mask]) >= 0.5 * total_flux:
            r_eff = radius
            break

    if r_eff == 0:
        r_eff = min(data.shape) // 4  # Default fallback

    return cy, cx, r_eff



# BAR DETECTION FEATURES (CENTRAL REGION ONLY)


def calc_bar_to_bulge_ratio_focused(data, cy, cx, r_eff):
    """
    FIXED: Bar/Bulge ratio in central region only
    """
    try:
        bar_search_radius = max(15, int(0.4 * r_eff))
        bulge_radius = max(5, int(0.12 * r_eff))

        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)

        # Bulge region
        bulge_mask = r < bulge_radius
        # Bar region (annulus)
        bar_region_mask = (r >= bulge_radius) & (r < bar_search_radius)

        if not np.any(bulge_mask) or not np.any(bar_region_mask):
            return 1.0

        bulge_flux = np.sum(data[bulge_mask])
        if bulge_flux < 1e-8:
            return 1.0

        # Extract bar region
        bar_data = np.zeros_like(data)
        bar_data[bar_region_mask] = data[bar_region_mask]

        # Calculate moments to measure elongation
        M = moments(bar_data, order=2)
        if M[0, 0] < 1e-8:
            return 1.0

        # Centroid of bar region
        x_bar = M[1, 0] / M[0, 0]
        y_bar = M[0, 1] / M[0, 0]

        # Central moments
        mu = moments_central(bar_data, y_bar, x_bar, order=2)
        if mu[0, 0] < 1e-8:
            return 1.0

        # Normalized moments
        mu20 = mu[2, 0] / mu[0, 0]
        mu02 = mu[0, 2] / mu[0, 0]
        mu11 = mu[1, 1] / mu[0, 0]

        # Eigenvalues for elongation
        term1 = (mu20 + mu02) / 2
        term2 = np.sqrt(((mu20 - mu02) / 2)**2 + mu11**2)

        lambda_max = term1 + term2
        lambda_min = term1 - term2

        if lambda_min < 1e-8:
            elongation = 3.0  # High elongation
        else:
            elongation = np.sqrt(lambda_max / lambda_min)

        # Clip elongation to reasonable range
        elongation = np.clip(elongation, 1.0, 5.0)

        # Bar flux weighted by elongation
        bar_flux = np.sum(bar_data) * (elongation / 2.0)

        ratio = bar_flux / bulge_flux

        return np.clip(ratio, 0.5, 10.0)

    except Exception as e:
        return 1.0


def calc_bar_fourier_strength_focused(data, cy, cx, r_eff):
    """
    Fourier m=2 mode strength in central region (bar signature)
    """
    try:
        bar_radius = max(15, int(0.4 * r_eff))
        inner_r = max(5, int(0.1 * r_eff))

        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)
        theta = np.arctan2(y - cy, x - cx)

        annulus = (r >= inner_r) & (r < bar_radius)

        if not np.any(annulus):
            return 0.0

        I = data[annulus]
        phi = theta[annulus]

        # m=2 Fourier mode
        A2_real = np.sum(I * np.cos(2 * phi))
        A2_imag = np.sum(I * np.sin(2 * phi))
        A0 = np.sum(I)

        if A0 < 1e-8:
            return 0.0

        A2 = np.sqrt(A2_real**2 + A2_imag**2) / A0

        return min(A2, 1.0)

    except:
        return 0.0


def calc_central_asymmetry_focused(data, cy, cx, r_eff, angle=180):
    """
    Asymmetry in central region only
    """
    try:
        central_radius = max(15, int(0.35 * r_eff))

        y_min = max(0, cy - central_radius)
        y_max = min(data.shape[0], cy + central_radius)
        x_min = max(0, cx - central_radius)
        x_max = min(data.shape[1], cx + central_radius)

        central = data[y_min:y_max, x_min:x_max]

        if central.size == 0:
            return 0.0

        # Rotate and compare
        rotated = ndimage.rotate(central, angle, reshape=False, order=1)

        diff = np.abs(central - rotated)
        asymmetry = np.sum(diff) / (2 * np.sum(np.abs(central)) + 1e-10)

        return min(asymmetry, 1.0)

    except:
        return 0.0



# UNBARRED-SPECIFIC FEATURES


def calc_nuclear_concentration_ratio(data, cy, cx, r_eff):
    """
    Unbarred spirals have stronger nuclear concentration
    """
    try:
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)

        nucleus_r = max(3, int(0.05 * r_eff))
        ring_outer = max(10, int(0.15 * r_eff))

        nucleus_mask = r <= nucleus_r
        ring_mask = (r > nucleus_r) & (r <= ring_outer)

        if not np.any(nucleus_mask) or not np.any(ring_mask):
            return 0.0

        nucleus_flux = np.sum(data[nucleus_mask])
        ring_flux = np.sum(data[ring_mask])

        if ring_flux < 1e-8:
            return 0.0

        ratio = nucleus_flux / ring_flux

        return min(ratio, 5.0)

    except:
        return 0.0


def calc_spiral_symmetry_score(data, cy, cx, r_eff):
    """
    Multi-mode Fourier analysis for spiral symmetry
    Unbarred: balanced m=2,3,4; Barred: m=2 dominant
    """
    try:
        inner_r = max(15, int(0.3 * r_eff))
        outer_r = max(30, int(0.8 * r_eff))

        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)
        theta = np.arctan2(y - cy, x - cx)

        spiral_region = (r >= inner_r) & (r < outer_r)

        if not np.any(spiral_region):
            return 0.0

        I = data[spiral_region]
        phi = theta[spiral_region]

        A0 = np.sum(I) + 1e-10

        # Multiple modes
        A2 = np.abs(np.sum(I * np.exp(2j * phi))) / A0
        A3 = np.abs(np.sum(I * np.exp(3j * phi))) / A0
        A4 = np.abs(np.sum(I * np.exp(4j * phi))) / A0

        # Balance score
        symmetry_balance = (A3 + A4) / (A2 + 1e-6)

        return min(symmetry_balance, 10.0)

    except:
        return 0.0


def calc_bar_ansae_test(data, cy, cx, r_eff):
    """
    Bar ansae: bright spots at bar ends
    Returns HIGH for unbarred, LOW for barred
    """
    try:
        test_radius = max(10, int(0.25 * r_eff))
        sample_size = max(3, test_radius // 3)

        # Sample 4 directions (horizontal, vertical, diagonals)
        directions = []

        # Horizontal
        if cy - sample_size >= 0 and cy + sample_size < data.shape[0]:
            if cx + test_radius + sample_size < data.shape[1]:
                h1 = data[cy-sample_size:cy+sample_size, cx+test_radius-sample_size:cx+test_radius+sample_size]
                directions.append(np.mean(h1))
            if cx - test_radius - sample_size >= 0:
                h2 = data[cy-sample_size:cy+sample_size, cx-test_radius-sample_size:cx-test_radius+sample_size]
                directions.append(np.mean(h2))

        # Vertical
        if cx - sample_size >= 0 and cx + sample_size < data.shape[1]:
            if cy + test_radius + sample_size < data.shape[0]:
                v1 = data[cy+test_radius-sample_size:cy+test_radius+sample_size, cx-sample_size:cx+sample_size]
                directions.append(np.mean(v1))
            if cy - test_radius - sample_size >= 0:
                v2 = data[cy-test_radius-sample_size:cy-test_radius+sample_size, cx-sample_size:cx+sample_size]
                directions.append(np.mean(v2))

        if len(directions) < 2:
            return 0.5

        max_dir = max(directions)
        mean_dir = np.mean(directions)

        # If one direction dominates = bar ansae
        if mean_dir < 1e-8:
            return 0.5

        ansae_score = 1.0 - min((max_dir / mean_dir - 1.0), 1.0)

        return max(0.0, min(ansae_score, 1.0))

    except:
        return 0.5


def calc_edge_on_indicator(data, cy, cx):
    """
    FIXED: Detect edge-on galaxies (elongated)
    """
    try:
        # Calculate moments for full galaxy
        M = moments(data, order=2)

        if M[0, 0] < 1e-8:
            return 0.0

        # Centroid
        xc = M[1, 0] / M[0, 0]
        yc = M[0, 1] / M[0, 0]

        # Central moments
        mu = moments_central(data, yc, xc, order=2)

        if mu[0, 0] < 1e-8:
            return 0.0

        # Normalized moments
        mu20 = mu[2, 0] / mu[0, 0]
        mu02 = mu[0, 2] / mu[0, 0]

        if mu02 < 1e-8 or mu20 < 1e-8:
            return 0.0

        # Axis ratio (always > 1)
        axis_ratio = max(mu20 / mu02, mu02 / mu20)

        # Edge-on if axis_ratio > 2.5
        edge_on_score = (axis_ratio - 1.0) / 5.0

        return max(0.0, min(edge_on_score, 1.0))

    except:
        return 0.0


# STANDARD MORPHOLOGY FEATURES


def calc_concentration_index(data, cy, cx):
    """Concentration C = 5 * log10(r80/r20)"""
    try:
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)

        total = np.sum(data)
        if total < 1e-8:
            return 0.0

        r20, r80 = 0, 0
        max_r = min(data.shape) // 2

        for radius in range(1, max_r):
            mask = r < radius
            flux_frac = np.sum(data[mask]) / total
            if flux_frac >= 0.2 and r20 == 0:
                r20 = radius
            if flux_frac >= 0.8:
                r80 = radius
                break

        if r20 == 0 or r80 == 0 or r20 >= r80:
            return 0.0

        C = 5 * np.log10(r80 / r20)
        return max(0.0, C)

    except:
        return 0.0


def calc_gini_coefficient(data):
    """Gini coefficient"""
    try:
        sorted_data = np.sort(data.flatten())
        sorted_data = sorted_data[sorted_data > 0]  # Remove zeros

        if len(sorted_data) == 0:
            return 0.0

        n = len(sorted_data)
        index = np.arange(1, n + 1)

        gini = (2 * np.sum(index * sorted_data)) / (n * np.sum(sorted_data)) - (n + 1) / n

        return max(0.0, min(gini, 1.0))

    except:
        return 0.0


def calc_ellipticity(data, cy, cx):
    """
    FIXED: Ellipticity from second moments
    """
    try:
        M = moments(data, order=2)

        if M[0, 0] < 1e-8:
            return 0.0

        xc = M[1, 0] / M[0, 0]
        yc = M[0, 1] / M[0, 0]

        mu = moments_central(data, yc, xc, order=2)

        if mu[0, 0] < 1e-8:
            return 0.0

        mu20 = mu[2, 0] / mu[0, 0]
        mu02 = mu[0, 2] / mu[0, 0]
        mu11 = mu[1, 1] / mu[0, 0]

        # Ellipticity formula
        numerator = np.sqrt((mu20 - mu02)**2 + 4 * mu11**2)
        denominator = mu20 + mu02 + 1e-10

        ellipticity = numerator / denominator

        return min(ellipticity, 1.0)

    except:
        return 0.0


def calc_R50_R90(data, cy, cx):
    """
    FIXED: Half-light and 90% light radii
    """
    try:
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)

        total = np.sum(data)
        if total < 1e-8:
            return 0, 0

        R50, R90 = 0, 0
        max_radius = min(data.shape) // 2

        for radius in range(1, max_radius):
            mask = r < radius
            flux_frac = np.sum(data[mask]) / total

            if flux_frac >= 0.5 and R50 == 0:
                R50 = radius
            if flux_frac >= 0.9:
                R90 = radius
                break

        # Set defaults if not found
        if R50 == 0:
            R50 = max_radius // 4
        if R90 == 0:
            R90 = max_radius // 2

        return R50, R90

    except:
        return 0, 0

# MAIN FEATURE EXTRACTION

def extract_galaxy_features(fits_path):
    """
    Extract ALL features with center-focused bar detection
    """
    try:
        with fits.open(fits_path) as hdul:
            data = hdul[0].data

        if data is None:
            return None

        # Normalize to [0, 1]
        data = data.astype(float)
        data = (data - np.min(data)) / (np.max(data) - np.min(data) + 1e-10)

        # Get center and effective radius
        cy, cx, r_eff = get_galaxy_center_and_radius(data)

        features = {}

        # === BAR FEATURES (CENTRAL ONLY) ===
        features['Bar_Bulge_Ratio_Focused'] = calc_bar_to_bulge_ratio_focused(data, cy, cx, r_eff)
        features['Bar_Fourier_Strength_Focused'] = calc_bar_fourier_strength_focused(data, cy, cx, r_eff)
        features['Central_Asymmetry_180'] = calc_central_asymmetry_focused(data, cy, cx, r_eff, 180)
        features['Central_Asymmetry_90'] = calc_central_asymmetry_focused(data, cy, cx, r_eff, 90)

        # === UNBARRED-SPECIFIC FEATURES ===
        features['Nuclear_Concentration_Ratio'] = calc_nuclear_concentration_ratio(data, cy, cx, r_eff)
        features['Spiral_Symmetry_Score'] = calc_spiral_symmetry_score(data, cy, cx, r_eff)
        features['Bar_Ansae_Test'] = calc_bar_ansae_test(data, cy, cx, r_eff)
        features['Edge_On_Indicator'] = calc_edge_on_indicator(data, cy, cx)

        # === STANDARD MORPHOLOGY ===
        features['Concentration'] = calc_concentration_index(data, cy, cx)
        R50, R90 = calc_R50_R90(data, cy, cx)
        features['R50'] = R50
        features['R90'] = R90
        features['Gini'] = calc_gini_coefficient(data)
        features['Ellipticity'] = calc_ellipticity(data, cy, cx)

        # === SIMPLE STATISTICS ===
        features['Mean_Intensity'] = np.mean(data)
        features['Std_Intensity'] = np.std(data)
        features['Skewness'] = skew(data.flatten())
        features['Kurtosis'] = kurtosis(data.flatten())

        # === DERIVED FEATURES ===
        features['Radial_Profile_Ratio'] = R90 / (R50 + 1e-6) if R50 > 0 else 0
        features['Effective_Radius'] = r_eff

        return features

    except Exception as e:
        print(f"Error processing {fits_path}: {e}")
        return None


def process_folder(folder_path, output_csv):
    """
    Process all FITS files in folder
    """
    print(f"\nProcessing folder: {folder_path}")

    fits_files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]
    print(f"Found {len(fits_files)} FITS files")

    all_features = []

    for idx, fits_file in enumerate(fits_files):
        if (idx + 1) % 50 == 0:
            print(f"  Processed {idx + 1}/{len(fits_files)} files...")

        fits_path = os.path.join(folder_path, fits_file)
        features = extract_galaxy_features(fits_path)

        if features is not None:
            features['filename'] = fits_file
            all_features.append(features)

    df = pd.DataFrame(all_features)

    # Reorder columns (filename first)
    cols = ['filename'] + [col for col in df.columns if col != 'filename']
    df = df[cols]

    df.to_csv(output_csv, index=False)
    print(f"\n✓ Saved features to {output_csv}")
    print(f"  Shape: {df.shape}")
    print(f"  Features: {len(df.columns) - 1}")

    # Quick stats check
    print(f"\n  Feature Value Ranges:")
    for col in df.columns:
        if col != 'filename':
            print(f"    {col:<35} min={df[col].min():.4f}, max={df[col].max():.4f}, mean={df[col].mean():.4f}")

    return df


if __name__ == "__main__":
    print("="*80)
    print("IMPROVED GALAXY FEATURE EXTRACTION (CENTER-FOCUSED)")
    print("="*80)

    train_folder = "train"
    test_folder = "test"

    if os.path.exists(train_folder):
        print(f"\n{'='*80}")
        print("EXTRACTING TRAINING SET FEATURES")
        print("="*80)
        train_df = process_folder(train_folder, "train_galaxy_features_improved.csv")
    else:
        print(f"\n⚠ Warning: {train_folder} not found!")

    if os.path.exists(test_folder):
        print(f"\n{'='*80}")
        print("EXTRACTING TEST SET FEATURES")
        print("="*80)
        test_df = process_folder(test_folder, "test_galaxy_features_improved.csv")
    else:
        print(f"\n⚠ Warning: {test_folder} not found!")

    print("\n" + "="*80)
    print("FEATURE EXTRACTION COMPLETE!")
    print("="*80)
    print("\nNext steps:")
    print("1. Check the CSV files to verify features look correct")
    print("2. Run the training script: python train_improved_classifier.py")
    print("="*80)

IMPROVED GALAXY FEATURE EXTRACTION (CENTER-FOCUSED)

EXTRACTING TRAINING SET FEATURES

Processing folder: train
Found 467 FITS files
  Processed 50/467 files...
  Processed 100/467 files...
  Processed 150/467 files...
  Processed 200/467 files...
  Processed 250/467 files...
  Processed 300/467 files...
  Processed 350/467 files...
  Processed 400/467 files...
  Processed 450/467 files...

✓ Saved features to train_galaxy_features_improved.csv
  Shape: (467, 20)
  Features: 19

  Feature Value Ranges:
    Bar_Bulge_Ratio_Focused             min=1.0000, max=1.0000, mean=1.0000
    Bar_Fourier_Strength_Focused        min=0.0036, max=0.4649, mean=0.1193
    Central_Asymmetry_180               min=0.0191, max=0.4697, mean=0.1409
    Central_Asymmetry_90                min=0.0288, max=0.5417, mean=0.1991
    Nuclear_Concentration_Ratio         min=0.0576, max=1.0040, mean=0.1302
    Spiral_Symmetry_Score               min=0.0618, max=10.0000, mean=1.1000
    Bar_Ansae_Test                 

In [ ]:
import os
import numpy as np
import pandas as pd
from astropy.io import fits
from scipy import ndimage
from scipy.stats import skew, kurtosis
from skimage.measure import moments, moments_central
from skimage.transform import resize
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# ADAPTIVE CENTER AND RADIUS DETECTION
# ============================================================================

def find_galaxy_center_and_radius(data, percentile=99.5):
    """
    Find galaxy center using brightest region and estimate effective radius
    Returns: (cy, cx, effective_radius)
    """
    # Threshold to find bright regions
    threshold = np.percentile(data, percentile)
    bright_mask = data > threshold

    if np.sum(bright_mask) == 0:
        # Fallback to geometric center
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        radius = min(data.shape) // 3
        return cy, cx, radius

    # Find center of mass of bright regions
    labeled, num_features = ndimage.label(bright_mask)

    if num_features == 0:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        radius = min(data.shape) // 3
        return cy, cx, radius

    # Get largest bright component (nucleus)
    sizes = ndimage.sum(bright_mask, labeled, range(num_features + 1))
    largest_component = np.argmax(sizes[1:]) + 1
    nucleus_mask = labeled == largest_component

    # Center of mass of nucleus
    cy, cx = ndimage.center_of_mass(data, nucleus_mask)
    cy, cx = int(cy), int(cx)

    # Estimate effective radius (radius containing 50% of light)
    y, x = np.ogrid[:data.shape[0], :data.shape[1]]
    r = np.sqrt((y - cy)**2 + (x - cx)**2)

    total_flux = np.sum(data)
    for radius in range(10, min(data.shape) // 2):
        mask = r < radius
        if np.sum(data[mask]) >= 0.5 * total_flux:
            break

    return cy, cx, radius


# ============================================================================
# BAR DETECTION (CENTRAL REGION ONLY)
# ============================================================================

def calc_bar_to_bulge_ratio_focused(data, cy, cx, r_eff):
    """
    Bar/Bulge ratio calculated ONLY in central region (< 0.4 * r_eff)
    Bars are central features, not outer spiral arms!
    """
    try:
        # Focus on INNER region only
        bar_search_radius = int(0.4 * r_eff)
        if bar_search_radius < 10:
            bar_search_radius = 10

        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)

        # Central circular region (bulge)
        bulge_radius = max(5, int(0.15 * r_eff))
        bulge_mask = r < bulge_radius

        # Bar search region (annulus around bulge)
        bar_region_mask = (r >= bulge_radius) & (r < bar_search_radius)

        if not np.any(bulge_mask) or not np.any(bar_region_mask):
            return 1.0

        bulge_flux = np.sum(data[bulge_mask])

        if bulge_flux < 1e-6:
            return 1.0

        # Look for elongated structure in bar region
        bar_data = data.copy()
        bar_data[~bar_region_mask] = 0

        # Calculate elongation using second moments
        M = moments(bar_data, order=2)
        if M[0, 0] < 1e-10:
            return 1.0

        # Centroid
        xc = M[1, 0] / M[0, 0]
        yc = M[0, 1] / M[0, 0]

        # Central moments
        mu = moments_central(bar_data, yc, xc, order=2)

        if mu[0, 0] < 1e-10:
            return 1.0

        # Normalized moments
        mu20_norm = mu[2, 0] / (mu[0, 0] + 1e-10)
        mu02_norm = mu[0, 2] / (mu[0, 0] + 1e-10)
        mu11_norm = mu[1, 1] / (mu[0, 0] + 1e-10)

        # Elongation measure
        lambda_plus = (mu20_norm + mu02_norm) / 2 + np.sqrt(((mu20_norm - mu02_norm) / 2)**2 + mu11_norm**2)
        lambda_minus = (mu20_norm + mu02_norm) / 2 - np.sqrt(((mu20_norm - mu02_norm) / 2)**2 + mu11_norm**2)

        if lambda_minus < 1e-10:
            elongation = 1.0
        else:
            elongation = np.sqrt(lambda_plus / (lambda_minus + 1e-10))

        # Bar flux weighted by elongation
        bar_flux = np.sum(bar_data) * elongation

        ratio = bar_flux / bulge_flux

        return np.clip(ratio, 0.5, 5.0)

    except Exception as e:
        return 1.0


def calc_bar_fourier_strength_focused(data, cy, cx, r_eff):
    """
    Fourier bar strength in CENTRAL region only (m=2 mode)
    """
    try:
        bar_radius = int(0.4 * r_eff)
        if bar_radius < 10:
            bar_radius = 10

        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)
        theta = np.arctan2(y - cy, x - cx)

        # Annular region where bars exist
        inner_r = int(0.1 * r_eff)
        annulus = (r >= inner_r) & (r < bar_radius)

        if not np.any(annulus):
            return 0.0

        I = data[annulus]
        phi = theta[annulus]

        # m=2 Fourier mode (bar signature)
        A2_real = np.sum(I * np.cos(2 * phi))
        A2_imag = np.sum(I * np.sin(2 * phi))
        A0 = np.sum(I)

        if A0 == 0:
            return 0.0

        A2 = np.sqrt(A2_real**2 + A2_imag**2) / A0

        return A2

    except:
        return 0.0


def calc_central_asymmetry_focused(data, cy, cx, r_eff, angle=180):
    """
    Asymmetry in CENTRAL region only - bars break symmetry centrally
    """
    try:
        central_radius = int(0.35 * r_eff)
        if central_radius < 10:
            return 0.0

        # Extract central region
        y_min = max(0, cy - central_radius)
        y_max = min(data.shape[0], cy + central_radius)
        x_min = max(0, cx - central_radius)
        x_max = min(data.shape[1], cx + central_radius)

        central = data[y_min:y_max, x_min:x_max]

        if central.size == 0:
            return 0.0

        # Rotate by angle
        rotated = ndimage.rotate(central, angle, reshape=False, order=1)

        # Asymmetry metric
        diff = np.abs(central - rotated)
        asymmetry = np.sum(diff) / (2 * np.sum(np.abs(central)) + 1e-10)

        return asymmetry

    except:
        return 0.0


# ============================================================================
# UNBARRED-SPECIFIC FEATURES
# ============================================================================

def calc_nuclear_concentration_ratio(data, cy, cx, r_eff):
    """
    Unbarred spirals have stronger nuclear concentration
    Barred galaxies redistribute mass to bar
    """
    try:
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)

        # Very central nucleus
        nucleus_r = max(3, int(0.05 * r_eff))
        nucleus_mask = r <= nucleus_r

        # Surrounding ring
        ring_inner = nucleus_r
        ring_outer = int(0.15 * r_eff)
        ring_mask = (r > ring_inner) & (r <= ring_outer)

        if not np.any(nucleus_mask) or not np.any(ring_mask):
            return 0.0

        nucleus_flux = np.sum(data[nucleus_mask])
        ring_flux = np.sum(data[ring_mask])

        ratio = nucleus_flux / (ring_flux + 1e-6)

        return ratio

    except:
        return 0.0


def calc_spiral_symmetry_score(data, cy, cx, r_eff):
    """
    Unbarred spirals have better m=2,3,4 symmetry balance
    Barred spirals dominated by m=2 (bar)
    """
    try:
        # Work in outer spiral region
        inner_r = int(0.3 * r_eff)
        outer_r = int(0.8 * r_eff)

        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)
        theta = np.arctan2(y - cy, x - cx)

        spiral_region = (r >= inner_r) & (r < outer_r)

        if not np.any(spiral_region):
            return 0.0

        I = data[spiral_region]
        phi = theta[spiral_region]

        # Calculate multiple Fourier modes
        A0 = np.sum(I) + 1e-10

        A2 = np.abs(np.sum(I * np.exp(2j * phi))) / A0
        A3 = np.abs(np.sum(I * np.exp(3j * phi))) / A0
        A4 = np.abs(np.sum(I * np.exp(4j * phi))) / A0

        # Unbarred: balanced multi-mode
        # Barred: m=2 dominates
        symmetry_balance = (A3 + A4) / (A2 + 1e-6)

        return symmetry_balance

    except:
        return 0.0


def calc_bar_ansae_test(data, cy, cx, r_eff):
    """
    Bar ansae: bright spots at bar ends (perpendicular positions)
    Low values = bar present, High values = no bar
    """
    try:
        test_radius = int(0.25 * r_eff)
        if test_radius < 5:
            return 1.0

        sample_size = max(3, test_radius // 3)

        # Sample 4 directions
        directions = []

        # Horizontal
        h1 = data[cy-sample_size:cy+sample_size, cx+test_radius-sample_size:cx+test_radius+sample_size]
        h2 = data[cy-sample_size:cy+sample_size, cx-test_radius-sample_size:cx-test_radius+sample_size]

        # Vertical
        v1 = data[cy+test_radius-sample_size:cy+test_radius+sample_size, cx-sample_size:cx+sample_size]
        v2 = data[cy-test_radius-sample_size:cy-test_radius+sample_size, cx-sample_size:cx+sample_size]

        for region in [h1, h2, v1, v2]:
            if region.size > 0:
                directions.append(np.mean(region))
            else:
                directions.append(0)

        if len(directions) == 0:
            return 1.0

        max_dir = max(directions)
        mean_dir = np.mean(directions)

        # If one direction dominates = bar ansae present
        ansae_score = 1 - (max_dir / (mean_dir + 1e-6) - 1)

        return np.clip(ansae_score, 0, 2)

    except:
        return 1.0


def calc_edge_on_indicator(data, cy, cx):
    """
    Detect edge-on galaxies (often misclassified as barred due to elongation)
    High values = edge-on
    """
    try:
        # Use central region
        size = min(data.shape[0], data.shape[1]) // 2
        y_min = max(0, cy - size)
        y_max = min(data.shape[0], cy + size)
        x_min = max(0, cx - size)
        x_max = min(data.shape[1], cx + size)

        central = data[y_min:y_max, x_min:x_max]

        if central.size == 0:
            return 0.0

        # Compute moments for entire galaxy
        mu = moments_central(central, central.shape[0]//2, central.shape[1]//2, order=2)

        if mu[0, 0] == 0:
            return 0.0

        mu20 = mu[2, 0] / mu[0, 0]
        mu02 = mu[0, 2] / mu[0, 0]

        # Very high elongation + thin = edge-on
        elongation = max(mu20, mu02) / (min(mu20, mu02) + 1e-6)

        # Edge-on if elongation > 3
        edge_on_score = max(0, elongation - 3) / 10

        return min(edge_on_score, 1.0)

    except:
        return 0.0


# ============================================================================
# OTHER STANDARD FEATURES (ADJUSTED FOR CENTER)
# ============================================================================

def calc_concentration_index(data, cy, cx):
    """Concentration C = 5 * log10(r80/r20)"""
    try:
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)

        total = np.sum(data)
        r20, r80 = 0, 0

        for radius in range(1, min(data.shape) // 2):
            mask = r < radius
            if np.sum(data[mask]) >= 0.2 * total:
                r20 = radius
                break

        for radius in range(r20, min(data.shape) // 2):
            mask = r < radius
            if np.sum(data[mask]) >= 0.8 * total:
                r80 = radius
                break

        if r20 == 0:
            return 0

        C = 5 * np.log10(r80 / r20) if r80 > 0 and r20 > 0 else 0
        return C
    except:
        return 0


def calc_gini_coefficient(data):
    """Gini coefficient - light distribution inequality"""
    try:
        sorted_data = np.sort(data.flatten())
        n = len(sorted_data)
        index = np.arange(1, n + 1)
        gini = (2 * np.sum(index * sorted_data)) / (n * np.sum(sorted_data)) - (n + 1) / n
        return gini
    except:
        return 0


def calc_ellipticity(data, cy, cx):
    """Ellipticity from second moments"""
    try:
        mu = moments_central(data, cy, cx, order=2)
        mu20 = mu[2, 0] / mu[0, 0]
        mu02 = mu[0, 2] / mu[0, 0]
        mu11 = mu[1, 1] / mu[0, 0]

        ellipticity = 1 - np.sqrt(((mu20 - mu02)**2 + 4*mu11**2) / ((mu20 + mu02)**2))
        return abs(ellipticity)
    except:
        return 0


def calc_R50_R90(data, cy, cx):
    """Half-light and 90% light radii"""
    try:
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)

        total = np.sum(data)
        R50, R90 = 0, 0

        for radius in range(1, min(data.shape) // 2):
            mask = r < radius
            flux = np.sum(data[mask])
            if flux >= 0.5 * total and R50 == 0:
                R50 = radius
            if flux >= 0.9 * total:
                R90 = radius
                break

        return R50, R90
    except:
        return 0, 0


# ============================================================================
# MAIN FEATURE EXTRACTION
# ============================================================================

def extract_galaxy_features(fits_path):
    """
    Extract ALL features with center-focused bar detection
    """
    try:
        with fits.open(fits_path) as hdul:
            data = hdul[0].data

        if data is None:
            return None

        # Normalize
        data = data.astype(float)
        data = (data - np.min(data)) / (np.max(data) - np.min(data) + 1e-10)

        # Find adaptive center and radius
        cy, cx, r_eff = find_galaxy_center_and_radius(data)

        # Initialize features dict
        features = {}

        # === BAR FEATURES (CENTRAL ONLY) ===
        features['Bar_Bulge_Ratio_Focused'] = calc_bar_to_bulge_ratio_focused(data, cy, cx, r_eff)
        features['Bar_Fourier_Strength_Focused'] = calc_bar_fourier_strength_focused(data, cy, cx, r_eff)
        features['Central_Asymmetry_180'] = calc_central_asymmetry_focused(data, cy, cx, r_eff, 180)
        features['Central_Asymmetry_90'] = calc_central_asymmetry_focused(data, cy, cx, r_eff, 90)

        # === UNBARRED-SPECIFIC FEATURES ===
        features['Nuclear_Concentration_Ratio'] = calc_nuclear_concentration_ratio(data, cy, cx, r_eff)
        features['Spiral_Symmetry_Score'] = calc_spiral_symmetry_score(data, cy, cx, r_eff)
        features['Bar_Ansae_Test'] = calc_bar_ansae_test(data, cy, cx, r_eff)
        features['Edge_On_Indicator'] = calc_edge_on_indicator(data, cy, cx)

        # === STANDARD MORPHOLOGY FEATURES ===
        features['Concentration'] = calc_concentration_index(data, cy, cx)
        R50, R90 = calc_R50_R90(data, cy, cx)
        features['R50'] = R50
        features['R90'] = R90
        features['Gini'] = calc_gini_coefficient(data)
        features['Ellipticity'] = calc_ellipticity(data, cy, cx)

        # === SIMPLE STATISTICS ===
        features['Mean_Intensity'] = np.mean(data)
        features['Std_Intensity'] = np.std(data)
        features['Skewness'] = skew(data.flatten())
        features['Kurtosis'] = kurtosis(data.flatten())

        # === DERIVED FEATURES ===
        features['Radial_Profile_Ratio'] = R90 / (R50 + 1e-6)
        features['Effective_Radius'] = r_eff

        return features

    except Exception as e:
        print(f"Error processing {fits_path}: {e}")
        return None


def process_folder(folder_path, output_csv):
    """
    Process all FITS files in folder
    """
    print(f"Processing folder: {folder_path}")

    fits_files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]
    print(f"Found {len(fits_files)} FITS files")

    all_features = []

    for idx, fits_file in enumerate(fits_files):
        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(fits_files)} files...")

        fits_path = os.path.join(folder_path, fits_file)
        features = extract_galaxy_features(fits_path)

        if features is not None:
            features['filename'] = fits_file
            all_features.append(features)

    df = pd.DataFrame(all_features)

    # Reorder columns
    cols = ['filename'] + [col for col in df.columns if col != 'filename']
    df = df[cols]

    df.to_csv(output_csv, index=False)
    print(f"\n✓ Saved features to {output_csv}")
    print(f"  Shape: {df.shape}")
    print(f"  Features: {len(df.columns) - 1}")

    return df


if __name__ == "__main__":
    # Process training and test sets
    print("="*80)
    print("IMPROVED GALAXY FEATURE EXTRACTION (CENTER-FOCUSED)")
    print("="*80)

    train_folder = "train"
    test_folder = "test"

    if os.path.exists(train_folder):
        train_df = process_folder(train_folder, "train_galaxy_features_improved.csv")
    else:
        print(f"Warning: {train_folder} not found!")

    if os.path.exists(test_folder):
        test_df = process_folder(test_folder, "test_galaxy_features_improved.csv")
    else:
        print(f"Warning: {test_folder} not found!")

    print("\n" + "="*80)
    print("FEATURE EXTRACTION COMPLETE!")
    print("="*80)

IMPROVED GALAXY FEATURE EXTRACTION (CENTER-FOCUSED)
Processing folder: train
Found 467 FITS files
Processed 50/467 files...
Processed 100/467 files...
Processed 150/467 files...
Processed 200/467 files...
Processed 250/467 files...
Processed 300/467 files...
Processed 350/467 files...
Processed 400/467 files...
Processed 450/467 files...

✓ Saved features to train_galaxy_features_improved.csv
  Shape: (467, 20)
  Features: 19
Processing folder: test
Found 117 FITS files
Processed 50/117 files...
Processed 100/117 files...

✓ Saved features to test_galaxy_features_improved.csv
  Shape: (117, 20)
  Features: 19

FEATURE EXTRACTION COMPLETE!
